# NYC Taxi — Bronze Ingestion
> **Generated by Forge SDK** · `forge generate bronze --dataset nyc_taxi --source open-datasets`

| | |
|---|---|
| **Source** | Azure Open Datasets · `wasbs://nyctlc@azureopendatastore.blob.core.windows.net/{taxi_type}/` |
| **Output** | `bronze/nyc_taxi/{taxi_type}/year={Y}/month={M:02d}/` |
| **Schedule** | Monthly via `nyc_taxi_bronze` DAG |
| **Env vars** | `TAXI_TYPE` · `PARTITION_YEAR` · `PARTITION_MONTH` · `FORGE_ENV` |

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  🔒  FORGE SDK — PREAMBLE — DO NOT EDIT                                ║
# ║  Regenerate with: forge sdk regen preamble                              ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import json
import os
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql import types as T

from forge_sdk.notebook import get_session
from forge_sdk.paths import bronze

spark, log = get_session(job_name="NycTaxiBronze")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  🔒  FORGE SDK — PERFORMANCE SETTINGS — DO NOT EDIT                    ║
# ║  Regenerate with: forge sdk regen perf                                  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
spark.conf.set("spark.sql.adaptive.enabled",                       "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled",    "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled",              "true")
spark.conf.set("spark.sql.shuffle.partitions",                     "48")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled",     "true")
spark.conf.set("spark.sql.parquet.vorder.enabled",                 "true")   # V-Order
spark.conf.set("spark.sql.autoBroadcastJoinThreshold",             str(50 * 1024 * 1024))
spark.conf.set("spark.sql.extensions",                             "io.delta.sql.DeltaSparkSessionExtension")
spark.conf.set("spark.sql.catalog.spark_catalog",                  "org.apache.spark.sql.delta.catalog.DeltaCatalog")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  🔒  FORGE SDK — PARAMETERS — DO NOT EDIT                              ║
# ║  Injected at runtime by Papermill / SparkApplication env vars           ║
# ║  Override values below for local interactive development only           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
TAXI_TYPE       = os.environ.get("TAXI_TYPE",        "yellow")
PARTITION_YEAR  = int(os.environ.get("PARTITION_YEAR",  "2023"))
PARTITION_MONTH = int(os.environ.get("PARTITION_MONTH", "1"))
FORGE_ENV       = os.environ.get("FORGE_ENV",        "dev")

STARTED_AT = datetime.now(timezone.utc)
log.info("params taxi_type=%s year=%d month=%d env=%s",
         TAXI_TYPE, PARTITION_YEAR, PARTITION_MONTH, FORGE_ENV)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  🔒  FORGE SDK — SOURCE READ — DO NOT EDIT                             ║
# ╚══════════════════════════════════════════════════════════════════════════╝
spark.conf.set(
    "fs.azure.account.auth.type.azureopendatastore.blob.core.windows.net", "None"
)

SRC = (
    f"wasbs://nyctlc@azureopendatastore.blob.core.windows.net"
    f"/{TAXI_TYPE}/puYear={PARTITION_YEAR}/puMonth={PARTITION_MONTH}/*.parquet"
)

raw = (
    spark.read
    .option("mergeSchema", "true")
    .parquet(SRC)
)

log.info("source_read path=%s schema_fields=%d", SRC, len(raw.schema.fields))
raw.printSchema()

---
## ✏️  BUSINESS LOGIC — EDIT THIS SECTION
> Apply any source-specific column selection, renames, type casts, or audit columns.
> The `raw` DataFrame is available. Assign your result to `df`.
> Do **not** modify cells above or below this section.
---

In [ ]:
# Add audit columns and taxi type tag — customise as needed
df = (
    raw
    .select(
        "*",
        F.current_timestamp().alias("_ingested_at"),
        F.input_file_name().alias("_source_file"),
        F.lit(TAXI_TYPE).alias("_taxi_type"),
    )
)

# During development — inspect a sample
# df.show(5, truncate=False)

---
## 🔒  SDK — DO NOT EDIT BELOW
---

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  🔒  FORGE SDK — VALIDATE & WRITE                                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝
row_count = df.count()
log.info("row_count taxi_type=%s year=%d month=%d count=%d",
         TAXI_TYPE, PARTITION_YEAR, PARTITION_MONTH, row_count)

if row_count == 0:
    log.warning("no_data — skipping write")
    raise SystemExit(0)

DEST = bronze(f"nyc_taxi/{TAXI_TYPE}/year={PARTITION_YEAR}/month={PARTITION_MONTH:02d}")

(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DEST)
)

log.info("write_complete dest=%s rows=%d", DEST, row_count)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  🔒  FORGE SDK — TRACKER — DO NOT EDIT                                 ║
# ║  Written irrespective of DQ outcome. Source of truth for pipeline runs. ║
# ╚══════════════════════════════════════════════════════════════════════════╝
tracker = {
    "version":      "v1",
    "job":          "NycTaxiBronze",
    "dataset":      f"bronze/nyc_taxi/{TAXI_TYPE}",
    "partition":    {"year": PARTITION_YEAR, "month": PARTITION_MONTH},
    "status":       "success",
    "rows_written": row_count,
    "started_at":   STARTED_AT.isoformat(),
    "completed_at": datetime.now(timezone.utc).isoformat(),
    "forge_env":    FORGE_ENV,
}

tracker_path = bronze(
    f"nyc_taxi/{TAXI_TYPE}/year={PARTITION_YEAR}/month={PARTITION_MONTH:02d}/_tracker/tracker.json"
)

jvm  = spark.sparkContext._jvm
conf = spark.sparkContext._jsc.hadoopConfiguration()
p    = jvm.org.apache.hadoop.fs.Path(tracker_path)
out  = p.getFileSystem(conf).create(p, True)
out.write(bytearray(json.dumps(tracker, indent=2).encode("utf-8")))
out.close()

log.info("tracker_emitted path=%s", tracker_path)
print(json.dumps(tracker, indent=2))